# 🍔 Food-101 Model Evaluation (Correct Preprocessing)

This notebook compares:
- Best Model (before fine-tuning)
- Final Fine-Tuned Model

Metrics:
- Top-1 Accuracy
- Top-5 Accuracy

Uses **TFDS Food-101** with **EfficientNet preprocessing** (CRITICAL FIX).

In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import pandas as pd
import os

from tensorflow.keras.applications.efficientnet import preprocess_input

In [ ]:
# ⚙️ Config
IMG_SIZE = 224
BATCH_SIZE = 32

BEST_MODEL_PATH = '/kaggle/input/best-model/keras/default/1/best_model.keras'
FINAL_MODEL_PATH = '/kaggle/input/final-model/keras/default/1/final_model.keras'

assert os.path.exists(BEST_MODEL_PATH)
assert os.path.exists(FINAL_MODEL_PATH)

print('✅ Model files found')

In [ ]:
# 📦 Load TFDS Food-101 (validation only)
val_ds = tfds.load(
    'food101',
    split='validation',
    as_supervised=True,
    shuffle_files=False
)

In [ ]:
# ✅ CRITICAL: SAME preprocessing used during training
def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = preprocess_input(image)  # 🔥 FIX THAT SAVES ACCURACY
    return image, label

val_ds = (
    val_ds
    .map(preprocess, num_parallel_calls=tf.data.AUTOTUNE)
    .batch(BATCH_SIZE)
    .prefetch(tf.data.AUTOTUNE)
)

In [ ]:
# 📥 Load models
best_model = tf.keras.models.load_model(BEST_MODEL_PATH)
final_model = tf.keras.models.load_model(FINAL_MODEL_PATH)

print('✅ Models loaded')

In [ ]:
# 🔍 Robust evaluation (metric-name safe)
def evaluate(model, name):
    values = model.evaluate(val_ds, verbose=1)
    metrics = dict(zip(model.metrics_names, values))

    return {
        'Model': name,
        'Loss': metrics.get('loss'),
        'Top-1 Accuracy': metrics.get('sparse_categorical_accuracy')
            or metrics.get('accuracy'),
        'Top-5 Accuracy': metrics.get('sparse_top_k_categorical_accuracy')
    }

In [ ]:
# 📊 Run evaluation
results = []
results.append(evaluate(best_model, 'Best Model'))
results.append(evaluate(final_model, 'Fine-Tuned Final Model'))

df = pd.DataFrame(results)
df

In [ ]:
# 💾 Save results
os.makedirs('/kaggle/working/results', exist_ok=True)

csv_path = '/kaggle/working/results/model_comparison.csv'
html_path = '/kaggle/working/results/model_comparison.html'

df.to_csv(csv_path, index=False)
df.to_html(html_path, index=False)

print('✅ Saved results:')
print(csv_path)
print(html_path)